# Clase 08 - Reto Final: Moneyball en la NHL

## Objetivo
Aplicar los conceptos de las 8 clases del diplomado para evaluar a la plantilla de los **New York Rangers** usando métricas avanzadas (Expected Goals, Corsi, Bloqueos). Tras identificar a los jugadores con peor rendimiento, propondremos **3 fichajes estrella** que ocupen esas posiciones exactas, maximizando el valor usando un presupuesto máximo de **$15 millones de dólares**.

## Dataset
Utilizaremos el dataset de Kaggle: `camnugent/predict-nhl-player-salaries`.

In [ ]:
!uv run kaggle datasets download -d camnugent/predict-nhl-player-salaries --unzip -p data/nhl_salaries

### 1. Cargar Datos y Análisis de Rendimiento Individual (NY Rangers)
Empezaremos evaluando el desempeño ofensivo y defensivo de la plantilla actual de los Rangers.

In [ ]:
import pandas as pd

df = pd.read_csv('data/nhl_salaries/train.csv', encoding='iso-8859-1')
df['Salary'] = pd.to_numeric(df['Salary'], errors='coerce')

nyr = df[df['Team'] == 'NYR'].copy()
nyr['CF%'] = (nyr['CF'] / (nyr['CF'] + nyr['CA'])) * 100
nyr['Net_TKA'] = nyr['TKA'] - nyr['GVA']
nyr['xG_Diff'] = nyr['G'] - nyr['ixG'] # Goles sobre lo esperado

print("--- ANÁLISIS OFENSIVO: GOLES vs EXPECTED GOALS (ixG) ---")
print("Jugadores rindiendo POR DEBAJO de sus goles esperados (mala definición/mala suerte):")
underperformers = nyr.sort_values('xG_Diff').head(3)
print(underperformers[['Last Name', 'Position', 'G', 'ixG', 'xG_Diff']])

print("\n--- ANÁLISIS DEFENSIVO: BLOQUEOS Y PÉRDIDAS ---")
print("Jugadores con más pérdidas netas (Takeaways vs Giveaways) y bajo CF%:")
defensive_liabilities = nyr[(nyr['Net_TKA'] < 0) & (nyr['CF%'] < 50)].sort_values('Net_TKA').head(3)
print(defensive_liabilities[['Last Name', 'Position', 'CF%', 'Net_TKA', 'iBLK']])

### 2. Identificación de Posiciones a Reforzar
Tomando a los jugadores menos eficientes de nuestros análisis anteriores, definiremos qué posiciones necesitamos buscar en el mercado.

In [ ]:
# Extraer las posiciones de los peores evaluados
positions_to_replace = list(set(underperformers['Position'].tolist() + defensive_liabilities['Position'].tolist()))
print("Posiciones prioritarias a reforzar basándonos en bajo rendimiento:", positions_to_replace)

### 3. Búsqueda de Fichajes Estrella ("Moneyball") Mapeados a las Necesidades
Sabiendo qué perfiles tienen problemas buscaremos tres fichajes (cubriendo idealmente las posiciones necesarias) que superen el 50% de CF% y tengan alta producción ofensiva (PTS > 40), maximizando el *Value_Per_Million* (Puntos multiplicados por 1.5 + CF% por cada millón de salario).

In [ ]:
targets = df[(df['Team'] != 'NYR') & (df['Salary'] > 0)].copy()
targets['CF%'] = (targets['CF'] / (targets['CF'] + targets['CA'])) * 100
targets['Value_Score'] = (targets['PTS'] * 1.5) + targets['CF%']
targets['Value_Per_Million'] = targets['Value_Score'] / (targets['Salary'] / 1000000)

# Filtrar solo jugadores de posiciones que necesitamos reemplazar
targets = targets[targets['Position'].isin(positions_to_replace)]

# Filtrar base de "Estrellas" (PTS > 40, CF% > 50)
stars = targets[(targets['PTS'] > 40) & (targets['CF%'] > 50)].copy()
best_stars = stars.sort_values(by='Value_Per_Million', ascending=False)

print("Top 10 Jugadores Disponibles en esas posiciones por Valor/Millón:")
print(best_stars[['First Name', 'Last Name', 'Position', 'Team', 'Salary', 'PTS', 'CF%', 'Value_Per_Million']].head(10))

### 4. Selección de los 3 Fichajes Estrella bajo Presupuesto ($15M)

In [ ]:
picked = []
total_salary = 0
budget = 15000000

# Seleccionamos los mejores asegurando que no nos pasemos del presupuesto
for idx, row in best_stars.iterrows():
    if len(picked) < 3 and (total_salary + row['Salary']) <= budget:
        picked.append(row)
        total_salary += row['Salary']

print("=== LOS 3 FICHAJES ESTRELLA IDEALES PARA CUBRIR NUESTRAS BAJAS ===")
for p in picked:
    print(f"Jugador: {p['First Name']} {p['Last Name']} | Pos: {p['Position']} | Equipo Actual: {p['Team']}")
    print(f"   Salario: ${p['Salary']:,.0f} | Puntos: {p['PTS']} | CF%: {p['CF%']:.2f}%\n")

print(f"💰 Salario Total Consumido: ${total_salary:,.0f} / ${budget:,.0f}")